# Dependencies


In [ ]:
import pandas as pd
import numpy as np
from category_encoders import target_encoder
import warnings
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")


# Load datasets
train_df = pd.read_csv("../artifacts/data/02-preprocessed/train_df.csv")
eval_df = pd.read_csv("../artifacts/data/02-preprocessed/eval_df.csv")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

%matplotlib inline
mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["figure.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.5
# Customize Seaborn Parameters
sns.set()
rc = {
    "font.family": ["serif"],
    "font.serif": "Times New Roman",
    "grid.color": "gainsboro",
    "grid.linestyle": "-",
}
sns.set_style(rc=rc)
sns.set_context("notebook", font_scale=0.8)

In [6]:
# dataset we'll be working with
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12460 entries, 0 to 12459
Data columns (total 28 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  12460 non-null  object 
 1   house_type           12460 non-null  object 
 2   bathrooms            12460 non-null  int64  
 3   bedrooms             12460 non-null  int64  
 4   price                12460 non-null  float64
 5   locality             12460 non-null  object 
 6   lat                  12460 non-null  float64
 7   lng                  12460 non-null  float64
 8   condition            12460 non-null  object 
 9   furnishing           12460 non-null  object 
 10  24_hour_electricity  12460 non-null  int64  
 11  air_conditioning     12460 non-null  int64  
 12  apartment            12460 non-null  int64  
 13  balcony              12460 non-null  int64  
 14  chandelier           12460 non-null  int64  
 15  dining_area          12460 non-null 

**The Key Rule:**

- Fit encoders/transformers on train only
- Appy the learned mappings to eval
- Cache mappings for faster inference

**Rational**
This prevents information of eval leaking into train to learn those patterns as well, by doing this, eval clearly remains unseen by the model. Hence it simulates real world performance.


# Price


In [ ]:
train_df["log_price"] = np.log(train_df["price"])
eval_df["log_price"] = np.log(eval_df["price"])

# Locality Features


In [ ]:
def get_stats(train_df, K=50):
    ELITE_AREAS = [
        "Airport Residential Area",
        "Cantonments",
        "Ridge",
        "Roman Ridge",
        "Labone",
    ]

    # Calculate raw neighborhood stats
    support = (
        train_df.groupby("locality")
        .agg(
            n_listings=("log_price", "size"),
            median_log=("log_price", "median"),
            q25_log=("log_price", lambda x: x.quantile(0.25)),
            q75_log=("log_price", lambda x: x.quantile(0.75)),
        )
        .reset_index()
    )

    global_ref = {
        "median": train_df["log_price"].median(),
        "q25": train_df["log_price"].quantile(0.25),
        "q75": train_df["log_price"].quantile(0.75),
    }

    stats_map = {}
    for _, row in support.iterrows():
        name = row["locality"]
        n = row["n_listings"]

        if name in ELITE_AREAS:
            w = 1.0
        else:
            w = n / (n + K)

        if w >= 0.75:
            tier, code = "A", 3
        elif w >= 0.5:
            tier, code = "B", 2
        elif w >= 0.25:
            tier, code = "C", 1
        else:
            tier, code = "D", 0

        # Save to dictionary
        stats_map[name] = {
            "loc_med_log": w * row["median_log"] + (1 - w) * global_ref["median"],
            "loc_q25_log": w * row["q25_log"] + (1 - w) * global_ref["q25"],
            "loc_q75_log": w * row["q75_log"] + (1 - w) * global_ref["q75"],
            "loc_trust_score": w,
            "loc_tier_code": code,
            "loc_tier_label": tier,
        }

    return stats_map, global_ref


def apply_features(target_df, stats_map, global_ref):
    new_rows = []

    default_stats = {
        "loc_med_log": global_ref["median"],
        "loc_q25_log": global_ref["q25"],
        "loc_q75_log": global_ref["q75"],
        "loc_trust_score": 0.0,
        "loc_tier_code": 0,
        "loc_tier_label": "D",
    }

    for locality in target_df["locality"]:
        stats = stats_map.get(locality, default_stats)
        new_rows.append(stats)

    return pd.concat([target_df.reset_index(drop=True), pd.DataFrame(new_rows)], axis=1)
